In [ ]:
from typing import TypedDict, List, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

In [ ]:
class AgentState(TypedDict):
    topic: str
    items: List[str]
    scores: Annotated[List[dict], operator.add]
    best: dict

In [ ]:
class WorkerState(TypedDict):
    item: str

In [ ]:
def generate_items(state: AgentState) -> AgentState:
    catalog = {
        "fruits": ["apple", "banana", "cherry", "date", "elderberry"],
        "animals": ["cat", "hippopotamus", "owl", "tiger"],
        "cities": ["Tokyo", "Lisbon", "Reykjavik", "Cairo"],
    }
    state["items"] = catalog.get(state["topic"], [])
    print(f"Topic: {state['topic']} -> {state['items']}")
    return state

In [ ]:
def score_item(state: WorkerState) -> AgentState:
    item = state["item"]
    score = sum(ord(c) for c in item.lower())
    print(f"  scoring {item:<14} -> {score}")
    return {"scores": [{"item": item, "score": score}]}

In [ ]:
def fan_out(state: AgentState):
    return [Send("score_item", {"item": x}) for x in state["items"]]

In [ ]:
def pick_best(state: AgentState) -> AgentState:
    state["best"] = max(state["scores"], key=lambda r: r["score"])
    print(f"Best: {state['best']}")
    return state

In [ ]:
graph = StateGraph(AgentState)

In [ ]:
graph.add_node("generate_items", generate_items)
graph.add_node("score_item", score_item)
graph.add_node("pick_best", pick_best)

graph.set_entry_point("generate_items")

graph.add_conditional_edges("generate_items", fan_out, ["score_item"])
graph.add_edge("score_item", "pick_best")

graph.set_finish_point("pick_best")

In [ ]:
test = graph.compile()

In [ ]:
from IPython.display import Image, display
display(Image(test.get_graph().draw_mermaid_png()))

In [ ]:
result = test.invoke({"topic": "fruits", "scores": []})

In [ ]:
result